# 0 · Import Contracts to Volume

This notebook takes a **zip file** of contract PDFs and loads the individual
files into a Unity Catalog **Volume**. That volume becomes the knowledge source
for the Vector Search index created in notebook `1`.

**What this notebook does**
1. Creates the target catalog / schema / `contracts` volume (if they don't exist)
2. Reads `contracts.zip` from the `contracts` volume root
3. Extracts every PDF into that same volume, then removes the zip
4. Verifies the file count

> Upload `contracts.zip` to the `contracts` volume (`ZIP_PATH` below) before
> running — via the Databricks UI (Catalog > Volume > Upload) or the CLI
> (`databricks fs cp contracts.zip dbfs:<ZIP_PATH>`). No separate staging
> volume is needed.

### Define Parameters

In [ ]:
dbutils.widgets.text("CATALOG_NAME", "workspace", "Catalog Name")
dbutils.widgets.text("SCHEMA_NAME", "retail_campaign", "Schema Name")
dbutils.widgets.text("VOLUME_NAME", "contracts", "Volume (holds the zip + extracted PDFs)")
dbutils.widgets.text("ZIP_FILENAME", "contracts.zip", "Zip File Name")

CATALOG_NAME = dbutils.widgets.get("CATALOG_NAME")
SCHEMA_NAME = dbutils.widgets.get("SCHEMA_NAME")
VOLUME_NAME = dbutils.widgets.get("VOLUME_NAME")
ZIP_FILENAME = dbutils.widgets.get("ZIP_FILENAME")

TARGET_DIR = f"/Volumes/{CATALOG_NAME}/{SCHEMA_NAME}/{VOLUME_NAME}"
ZIP_PATH = f"{TARGET_DIR}/{ZIP_FILENAME}"

print("Contracts volume dir :", TARGET_DIR)
print("Zip path             :", ZIP_PATH)

### Create Catalog, Schema, and Volume

In [ ]:
spark.sql(f"CREATE CATALOG IF NOT EXISTS {CATALOG_NAME}")
spark.sql(f"CREATE SCHEMA IF NOT EXISTS {CATALOG_NAME}.{SCHEMA_NAME}")
spark.sql(f"CREATE VOLUME IF NOT EXISTS {CATALOG_NAME}.{SCHEMA_NAME}.{VOLUME_NAME}")
print("Catalog, schema, and volume are ready.")

### Extract the Zip into the Volume, then Remove It

The PDFs land directly in the `contracts` volume. We delete the zip afterwards so
notebook `1` only sees PDF files when it parses the volume.

In [ ]:
import os
import zipfile

assert os.path.exists(ZIP_PATH), (
    f"Zip not found at {ZIP_PATH}. Upload '{ZIP_FILENAME}' to the {VOLUME_NAME} volume first."
)

extracted = []
with zipfile.ZipFile(ZIP_PATH, "r") as zf:
    for member in zf.namelist():
        # skip directories and hidden/system files
        filename = os.path.basename(member)
        if not filename or filename.startswith(".") or member.endswith("/"):
            continue
        target_path = os.path.join(TARGET_DIR, filename)
        with zf.open(member) as src, open(target_path, "wb") as dst:
            dst.write(src.read())
        extracted.append(filename)

# Remove the zip so only PDFs remain in the volume
os.remove(ZIP_PATH)
print(f"Extracted {len(extracted)} files into {TARGET_DIR} and removed the zip")

### Verify

In [ ]:
files = [f for f in os.listdir(TARGET_DIR) if f.lower().endswith(".pdf")]
print(f"{len(files)} PDF files now in {TARGET_DIR}")
for f in sorted(files)[:5]:
    print("  ", f)
print("   ...")